Este notebook irá montar um tabela com dados dos Municipios Brasileiros e a quantidade da poplulação por raça e cor

Tabela 9605 - População residente, por cor ou raça, nos Censos Demográficos (Vide Notas)

- Amarela
- Branca
- Indigena
- Parda
- Preta

Link com dados do Sidra<br>
https://apisidra.ibge.gov.br/values/t/9605/n1/all/n6/1200203,1100346,1100072/v/allxp/p/last%201/c86/allxt


In [ ]:
import os, sys, requests
from pyspark.sql import functions as F

In [ ]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("IBGE_raca_cor")


Recupera as informações sobre raça e cor da população - IBGE via API SIDRA. 

Tabela 9605 - População residente, por cor ou raça, nos Censos Demográficos (Vide Notas)



In [ ]:
url_populacao_urbana_rural = "https://apisidra.ibge.gov.br/values/t/9605/n6/all/v/allxp/p/last%201/c86/allxt"
try:
    # 1. Faz a requisição HTTP para a API do IBGE
    response = requests.get(url_populacao_urbana_rural)
    response.raise_for_status()  # Garante que a requisição funcionou (Status 200)

    # 2. Converte a resposta bruta para o formato JSON
    populacao_raca_cor = response.json()

    print(len(populacao_raca_cor))
except Exception as e:
    print(e)    

In [ ]:


cabecalho    = populacao_raca_cor[0]
linhas_dados = populacao_raca_cor[1:]

mapeamento_colunas = {k: v for k, v in cabecalho.items()}
dados_processados = []
for linha in linhas_dados:
    nova_linha = {mapeamento_colunas[chave]: valor for chave, valor in linha.items() if chave in mapeamento_colunas}
    dados_processados.append(nova_linha)

df_populacao_raca_cor = spark.createDataFrame(dados_processados)



In [ ]:
df_populacao_raca_cor.printSchema()
df_populacao_raca_cor.show(10, False)

In [ ]:
# Separa somente as colunas desejadas.
df_populacao_raca_cor = \
    df_populacao_raca_cor.select(
         F.col("Ano").alias("ano")
        ,F.col("Município (Código)").alias("codigo_municipio")
        ,F.col("Município").alias("nome_municipio")
        ,F.col("Cor ou raça (Código)").alias("codigo_cor_raca")
        ,F.col("Cor ou raça").alias("descricao_cor_raca")
        ,F.col("Valor").alias("quantidade_populacao_cor_raca"))


In [ ]:
df_populacao_raca_cor.show(10, False)

### Faz a transposição das informações de linhas para colunas 

In [ ]:
df_populacao_raca_cor.select('descricao_cor_raca').dropDuplicates().show()

In [ ]:
df_populacao_raca_cor.createOrReplaceTempView("temp_municipios_cor_raca")
query = \
    """Select ano, codigo_municipio, descricao_cor_raca, quantidade_populacao_cor_raca
         from temp_municipios_cor_raca
    """
df_populacao_raca_cor_ = spark.sql(query)

# df_municipio_populacao_urbana_rural.printSchema()

# df_municipio_populacao_urbana_rural.show(10,False)

# Executando o Pivot para criar as colunas "Urbana" e "Rural" com os valores correspondentes
df_populacao_cor_raca_pivot = \
    df_populacao_raca_cor_ \
        .groupBy("ano", "codigo_municipio" ) \
        .pivot("descricao_cor_raca", ["Amarela", "Branca", "Indígena", "Parda", "Preta"]) \
        .agg(F.first("quantidade_populacao_cor_raca"))

# Exibindo o resultado
df_populacao_cor_raca_pivot.show(truncate=False)


In [ ]:
df_populacao_cor_raca_pivot.count()

In [ ]:
df_populacao_cor_raca_pivot.count()